<a href="https://colab.research.google.com/github/snxly/colab/blob/master/GPT_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building GPT-1
1. GPT-1 means basic Transformer with decoder only
2. use tiny-shakespere as dataset
3. character-level

# 第一步 准备数据
## 下载文件，加载文件

In [ ]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-04-09 02:42:09--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-04-09 02:42:09 (21.7 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [ ]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print('size of text', len(text))
print(text[:100])

size of text 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


## 准备 encoder 和 decoder
chars, vocab_size, stoi, itos, encode, decode

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("".join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [ ]:
stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}
# print(stoi)
# print(itos)
encode = lambda str: [stoi[char] for char in str]
decode = lambda arr: ''.join([itos[id] for id in arr])
str = 'abc'
str_enc = encode(str)
str_dec = decode(str_enc)
print(str_enc, str_dec)

[39, 40, 41] abc


# 第二步 准备dataset
## 首先，把text整个encode，并转成 torch.tensor

In [ ]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


## splits

In [ ]:
n = int(0.9 * len(text))
train_split = data[:n]
val_split = data[n:]
print(len(train_split), len(val_split))

1003854 111540


## get_batch

In [ ]:
batch_size = 4
block_size = 8

def get_batch(split):
  data = train_split if split == 'train' else val_split
  # 生成随机数。
  # 随机数的范围是[0,len(data) - block_size),这样连续取出 block_size + 1 个数以后，最后一个数的范围是 [block_size + 1, len(data))
  # 取 block_size + 1 个数的原因是，为了获取最后一个x的下一位
  # 第二个参数表示返回的形态，不是一个数，而是一个shape=[batch_size]的tensor
  ix = torch.randint(len(data) - block_size, (batch_size,))
  # 取出block_size + 1个数，前面block_size个数作为x，往后错一位作为y
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  # x,y 的shape都是 (batch_size, block_size)
  return x, y

# test code, print get_batch result by iterating x and y
xb, yb = get_batch('train')
for i in range(batch_size):
  for j in range(block_size):
    context = xb[i, :j+1]
    target = yb[i, j]
    print(f'when we input {context.tolist()}, target is {target}')


when we input [52], target is 43
when we input [52, 43], target is 57
when we input [52, 43, 57], target is 58
when we input [52, 43, 57, 58], target is 63
when we input [52, 43, 57, 58, 63], target is 1
when we input [52, 43, 57, 58, 63, 1], target is 39
when we input [52, 43, 57, 58, 63, 1, 39], target is 52
when we input [52, 43, 57, 58, 63, 1, 39, 52], target is 42
when we input [13], target is 25
when we input [13, 25], target is 21
when we input [13, 25, 21], target is 24
when we input [13, 25, 21, 24], target is 24
when we input [13, 25, 21, 24, 24], target is 27
when we input [13, 25, 21, 24, 24, 27], target is 10
when we input [13, 25, 21, 24, 24, 27, 10], target is 0
when we input [13, 25, 21, 24, 24, 27, 10, 0], target is 27
when we input [43], target is 1
when we input [43, 1], target is 61
when we input [43, 1, 61], target is 53
when we input [43, 1, 61, 53], target is 56
when we input [43, 1, 61, 53, 56], target is 52
when we input [43, 1, 61, 53, 56, 52], target is 1
whe

# 第三步 定义model
## 先从简单的开始，定义一个 BigramLanguageModel

In [ ]:
# 先写测试代码，确定等下准备怎么用这个 model

# 1. 使用 get_batch 获取 x, y
xb, yb = get_batch('train')
# 2. 初始化model
# model = BigramLanuageModel(vocab_size)
# 3. 执行forward，返回logits和loss
logits, loss = model(xb, yb)
# print logits and loss
print(f'logits.shape = {logits.shape}, loss={loss}')

# 4. 定义一个generate函数，用于inference
input = torch.zeros((1, 1), dtype=torch.long)
max_tokens = 1000
out = model.generate(input, max_tokens)
# Q：为什么要取 out[0]? generate返回值的格式是什么？
# A: input的格式是(B, T), generate的返回值格式也是 (B, T)，虽然B=1, 但是也占着一个维度
# 所以out[0]的意思就是，把唯一的这个sample取出来
out_str = decode(out[0].tolist())
print(f'generate result: {out_str}')


logits.shape = torch.Size([32, 8, 65]), loss=2.007361888885498
generate result: 
Shavir
The well soo herefulouring;
Be 
For in fre,
AweR I AWhtiled heat!
Offe to evey be what deeraous prists bufunto prewes it pourse suaght!

DADZE:
Whereab; kinkerI:
'To
Ris toobling of ofsell it a hes.

DUKE VARDSARD OMBNJY:
Yo ir aglant.

I sto-t wall noth it that cour way quries brow, Cowen.

TUnIDG RICIOLAT:
Arbey, prot for thou desp, of ort hather,
So mulce in wart srome; eagsen. Vord.
-Werars your, in coud;
That, Cem conote. Tandek,
To
Ford I her be mallough? 'To hend Carpie,
Qhall my conses;
Evich
There have and in ye in thighting thus. I If deret intre, Froawmer chore exers kne not.

CIOLUS:
You for by faliom enfatword,
Sing it age, his canougng;
Couch youn.
To
Lays recees.

KKINUCETY THANubeees ceamn her
fatay commest
Mared Jup cears
Im dleay she; noint:
Wout your and nens thou bithne;
Loose is
The be
Dixt.
Bot: jow her.

MINCELOLUS:
Shave minch cobled what rear loaral. I Lorrthare
And esem's h

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanuageModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

  def forward(self, idx, target=None):
    # idx, target 的shape是 (B, T)
    logits = self.token_embedding_table(idx) # (B, T, C) 即 (B, T, vocab_size)

    if target is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits_view = logits.view(B*T, C)
      target_view = target.view(B*T)
      # todo: 这里进行reshape是因为，F.cross_entropy的限制
      # A: 我们的数据的维度是 (B, T, C)，C是第三个维度，但是 F.cross_entropy 希望C是第二个维度
      loss = F.cross_entropy(logits_view, target_view)
    return logits, loss

  # input的shape是(1, 1), 代表batch=1， T=1
  def generate(self, input, max_tokens):
    context = input
    for i in range(max_tokens):
      # 虽然没用到loss，但是这里也得写，否则logits就变成(logits，loss)这个tuple了
      logits, loss = self(context)
      # 只关注T维度的最后一个
      logits = logits[:,-1,:] # 变成(B, C)
      # 在最后一个维度，即 vocab_size的维度上做 softmax
      probs = F.softmax(logits, dim=-1)
      # 使用multinomial 进行采样，获取下一个token
      next_token = torch.multinomial(probs, num_samples=1) # (B, 1)
      # 加到最后去，变成 (B, T+1)
      context = torch.cat((context, next_token), dim=1)
    return context


## 一个完整的训练过程

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

batch_size = 32
iterations = 1000

for i in range(iterations):
  xb, yb = get_batch('train')
  optimizer.zero_grad()
  logits, loss = model(xb, yb)
  loss.backward()
  optimizer.step()
  if i % 100 == 0:
    print(f'Step = {i}, loss={loss}')

Step = 0, loss=2.164642095565796
Step = 100, loss=2.0147976875305176
Step = 200, loss=2.0950582027435303
Step = 300, loss=2.0907647609710693
Step = 400, loss=2.101785898208618
Step = 500, loss=2.2625749111175537
Step = 600, loss=2.0734477043151855
Step = 700, loss=2.2004668712615967
Step = 800, loss=2.012363910675049
Step = 900, loss=2.021479845046997


# 第四部分 进入 Transformer
## 首先，我们聊 Self-Attention，通过 Head class来定义 self-attention 怎么计算


In [ ]:
class Head(nn.Module):
  def __init__(self, n_embed):
    super().__init__()
    # One-Head
    self.head_size = n_embed
    # 定义 q,k,v 的 projection
    self.q_proj = nn.Linear(n_embed, self.head_size, bias=False)
    self.k_proj = nn.Linear(n_embed, self.head_size, bias=False)
    self.v_proj = nn.Linear(n_embed, self.head_size, bias=False)
    # 当然还需要回来的projection。记错了，不需要这个。只需要concat，不需要project
    # 需要的是用register_buffer tirl (T, T)
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

  def forward(self, idx):
    # 这里负责 self-attention 的计算
    # 1. project q, k, v (B, T, n_embed) -> (B, T, head_size)
    B, T, n_embed = idx.shape
    q = self.q_proj(idx)
    k = self.k_proj(idx)
    v = self.v_proj(idx)
    # 2. 计算weight, 别忘了scale
    # B, T, head_size @ B, head_size, T => B, T, T
    weight = q @ k.transpose(-2, -1) * self.head_size ** -0.5
    # 3. 处理weight， 先用tril做mask，再softmax
    tril = self.tril[:T, :T]
    weight = weight.masked_fill(tril == 0, float('-inf'))
    # weight是 TxT 的矩阵，在第二个维度上（每一行）做softmax
    weight = F.softmax(weight, dim=-1)

    result = weight @ v
    return result


## 把 self-attention 集成到 model中
1. 引入 `n_embed`, 统一模型中的向量长度。把token_embedding改成两部分，(vocab_size, vocab_size) => (vocab_size, n_embed) + (n_embed, vocab_size) (lm_head) lm_head在self-attention结果上执行
2. 因为 self-attention 没有position信息，所以我们需要加一个 position_embedding
3. 集成 self-attention class。
4. 现在forward的数据流就变成 （token_emb+pos_emb） -> self_attention_head -> lm_head
5. 需要修改generate去截断context的最后block_size个token，因为 position_embedding只能处理 block_size 个token

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

n_embed = 32

class BigramLanuageModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
    self.sa_head = Head(n_embed)
    self.position_embedding_table = nn.Embedding(block_size, n_embed)
    self.lm_head = nn.Linear(n_embed, vocab_size)

  def forward(self, idx, target=None):
    # idx, target 的shape是 (B, T)
    B, T = idx.shape
    token_emb = self.token_embedding_table(idx) # (B, T, C) 即 (B, T, n_embed)
    pos_emb = self.position_embedding_table(torch.arange(T)) # (T, n_embed)
    x = token_emb + pos_emb

    x = self.sa_head(x)
    logits = self.lm_head(x)

    if target is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits_view = logits.view(B*T, C)
      target_view = target.view(B*T)
      # todo: 这里进行reshape是因为，F.cross_entropy的限制
      # A: 我们的数据的维度是 (B, T, C)，C是第三个维度，但是 F.cross_entropy 希望C是第二个维度
      loss = F.cross_entropy(logits_view, target_view)
    return logits, loss

  # input的shape是(1, 1), 代表batch=1， T=1
  def generate(self, input, max_tokens):
    result = input
    for i in range(max_tokens):
      # 虽然没用到loss，但是这里也得写，否则logits就变成(logits，loss)这个tuple了
      # 截断，因为position_embedding只能处理 blocl_size个数据
      context = result[:,-block_size:]
      logits, loss = self(context)
      # 只关注T维度的最后一个
      logits = logits[:,-1,:] # 变成(B, C)
      # 在最后一个维度，即 vocab_size的维度上做 softmax
      probs = F.softmax(logits, dim=-1)
      # 使用multinomial 进行采样，获取下一个token
      next_token = torch.multinomial(probs, num_samples=1) # (B, 1)
      # 加到最后去，变成 (B, T+1)
      result = torch.cat((result, next_token), dim=1)
    return result


## 接下来，把single-head self-attention 改成 multi-head self-attention
1. 把 Head 里的单头临时代码 `self.head_size = n_embed` 去掉
2. 创建 MultiHeadAttention Class
3. 在Model中使用 MultiHeadAttention

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

# 1.修改 Head 里初始化 self.head_size 的部分
class Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    # One-Head
    # self.head_size = n_embed
    # 定义 self.head_size, 因为在 forward 里的scale部分需要用到
    self.head_size = head_size
    # 定义 q,k,v 的 projection
    self.q_proj = nn.Linear(n_embed, self.head_size, bias=False)
    self.k_proj = nn.Linear(n_embed, self.head_size, bias=False)
    self.v_proj = nn.Linear(n_embed, self.head_size, bias=False)
    # 当然还需要回来的projection。记错了，不需要这个。只需要concat，不需要project
    # 需要的是用register_buffer tirl (T, T)
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

  def forward(self, idx):
    # 这里负责 self-attention 的计算
    # 1. project q, k, v (B, T, n_embed) -> (B, T, head_size)
    B, T, n_embed = idx.shape
    q = self.q_proj(idx)
    k = self.k_proj(idx)
    v = self.v_proj(idx)
    # 2. 计算weight, 别忘了scale
    # B, T, head_size @ B, head_size, T => B, T, T
    weight = q @ k.transpose(-2, -1) * self.head_size ** -0.5
    # 3. 处理weight， 先用tril做mask，再softmax
    tril = self.tril[:T, :T]
    weight = weight.masked_fill(tril == 0, float('-inf'))
    # weight是 TxT 的矩阵，在第二个维度上（每一行）做softmax
    weight = F.softmax(weight, dim=-1)

    result = weight @ v
    return result

# 2. 定义 MultiHeadAttention
class MultiHeadAttention(nn.Module):
  # n_head * head_size = n_embed
  def __init__(self, n_head, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for i in range(n_head)])

  def forward(self, x):
    headResult = [head(x) for head in self.heads]
    # head shape is (B, T, head_size)
    # 在最后一个维度做cat，所以result的维度是 (B, T, n_embed)
    result = torch.cat(headResult, dim=-1)
    return result

# 3. 在 Model 中,把Head替换成 MultiHeadAttention
n_embed = 32
head_size = 8
n_head = n_embed // head_size
print(f'n_head = {n_head}')

class BigramLanuageModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
    #self.sa_head = Head(n_embed)
    self.mh_attention = MultiHeadAttention(n_head, head_size)
    self.position_embedding_table = nn.Embedding(block_size, n_embed)
    self.lm_head = nn.Linear(n_embed, vocab_size)

  def forward(self, idx, target=None):
    # idx, target 的shape是 (B, T)
    B, T = idx.shape
    token_emb = self.token_embedding_table(idx) # (B, T, C) 即 (B, T, n_embed)
    pos_emb = self.position_embedding_table(torch.arange(T)) # (T, n_embed)
    x = token_emb + pos_emb

    # x = self.sa_head(x)
    x = self.mh_attention(x)
    logits = self.lm_head(x)

    if target is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits_view = logits.view(B*T, C)
      target_view = target.view(B*T)
      # todo: 这里进行reshape是因为，F.cross_entropy的限制
      # A: 我们的数据的维度是 (B, T, C)，C是第三个维度，但是 F.cross_entropy 希望C是第二个维度
      loss = F.cross_entropy(logits_view, target_view)
    return logits, loss

  # input的shape是(1, 1), 代表batch=1， T=1
  def generate(self, input, max_tokens):
    result = input
    for i in range(max_tokens):
      # 虽然没用到loss，但是这里也得写，否则logits就变成(logits，loss)这个tuple了
      # 截断，因为position_embedding只能处理 blocl_size个数据
      context = result[:,-block_size:]
      logits, loss = self(context)
      # 只关注T维度的最后一个
      logits = logits[:,-1,:] # 变成(B, C)
      # 在最后一个维度，即 vocab_size的维度上做 softmax
      probs = F.softmax(logits, dim=-1)
      # 使用multinomial 进行采样，获取下一个token
      next_token = torch.multinomial(probs, num_samples=1) # (B, 1)
      # 加到最后去，变成 (B, T+1)
      result = torch.cat((result, next_token), dim=1)
    return result


n_head = 4


## copy之前的测试代码， 简单测试一下

In [ ]:
# 1. 使用 get_batch 获取 x, y
xb, yb = get_batch('train')
# 2. 初始化model
#model = BigramLanuageModel(vocab_size)
# 3. 执行forward，返回logits和loss
logits, loss = model(xb, yb)
# print logits and loss
print(f'logits.shape = {logits.shape}, loss={loss}')

# 4. 定义一个generate函数，用于inference
input = torch.zeros((1, 1), dtype=torch.long)
max_tokens = 1000
out = model.generate(input, max_tokens)
# Q：为什么要取 out[0]? generate返回值的格式是什么？
# A: input的格式是(B, T), generate的返回值格式也是 (B, T)，虽然B=1, 但是也占着一个维度
# 所以out[0]的意思就是，把唯一的这个sample取出来
out_str = decode(out[0].tolist())
print(f'generate result: {out_str}')


logits.shape = torch.Size([32, 8, 65]), loss=2.049435615539551
generate result: 
Ichy soord and band, shoust whe the itsend houm's him it arise filf aly.

RULOLAS EDWARD YICISSSWARUS:
And wheare in mary how, wout the-do the of acm your rop! wore: so;
And cidr in as we itthere coosshong viche his prie to'd him.
CLTUCHERS Luons man:
No; I his place bet thou helives:
Obitng.

MERMER:

HARD FIVNCLLINTAMESY:
Gom'd!

HENRYBENILORD:
Herul fir of As tovetrus, how nothers. I the thy of is and rofad to cown this him, if, the highit o; the still of shairs--EL:
SARGETII:
PULINCW:
Shat appillimed
a the by my a that oHhand thou a taklll But herey. shour pas gry for, it Rut Pand binde clier Londeran?

PPORMES:
ObI

BORLABECAS:
TRuN in Shall:
Wechat as so supochat horees?

SAUSRIS:
Bod swall spelice?
That witheld.

PLIVO:
Shears:
Dothould of him nogt thusis:
What mie, deave for say goon it not the lied knociltar;: a nows.

PUIUPELES:
Should's shom.

HENMENS:
My
The atseld que hothert;
Hiced
Thoust the

In [ ]:
## Multi-Head Attention 完成以后，添加 FeedForward 部分

In [ ]:
# 1. 定义 FeedForward
class FeedForward(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embed, 4 * n_embed),
        nn.ReLU(),
        nn.Linear(4 * n_embed, n_embed),
    )

  def forward(self, x):
    return self.net(x)

# 2. 把FeedForward 加入 Model
class BigramLanuageModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
    #self.sa_head = Head(n_embed)
    self.mh_attention = MultiHeadAttention(n_head, head_size)
    self.ffw = FeedForward()
    self.position_embedding_table = nn.Embedding(block_size, n_embed)
    self.lm_head = nn.Linear(n_embed, vocab_size)

  def forward(self, idx, target=None):
    # idx, target 的shape是 (B, T)
    B, T = idx.shape
    token_emb = self.token_embedding_table(idx) # (B, T, C) 即 (B, T, n_embed)
    pos_emb = self.position_embedding_table(torch.arange(T)) # (T, n_embed)
    x = token_emb + pos_emb

    # x = self.sa_head(x)
    x = self.mh_attention(x)
    x = self.ffw(x)
    logits = self.lm_head(x)

    if target is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits_view = logits.view(B*T, C)
      target_view = target.view(B*T)
      # todo: 这里进行reshape是因为，F.cross_entropy的限制
      # A: 我们的数据的维度是 (B, T, C)，C是第三个维度，但是 F.cross_entropy 希望C是第二个维度
      loss = F.cross_entropy(logits_view, target_view)
    return logits, loss

  # input的shape是(1, 1), 代表batch=1， T=1
  def generate(self, input, max_tokens):
    result = input
    for i in range(max_tokens):
      # 虽然没用到loss，但是这里也得写，否则logits就变成(logits，loss)这个tuple了
      # 截断，因为position_embedding只能处理 blocl_size个数据
      context = result[:,-block_size:]
      logits, loss = self(context)
      # 只关注T维度的最后一个
      logits = logits[:,-1,:] # 变成(B, C)
      # 在最后一个维度，即 vocab_size的维度上做 softmax
      probs = F.softmax(logits, dim=-1)
      # 使用multinomial 进行采样，获取下一个token
      next_token = torch.multinomial(probs, num_samples=1) # (B, 1)
      # 加到最后去，变成 (B, T+1)
      result = torch.cat((result, next_token), dim=1)
    return result

# 初始化一个model，用于训练和evaluate
model = BigramLanuageModel(vocab_size)

## 定义 Block class， 封装 Multi-Head Attention 和 FeedForward
这里把之前定义过的 class 都放在一起

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

# hyper-parameters
n_embed = 32
head_size = 8
n_head = n_embed // head_size
print(f'n_head = {n_head}')

class Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    # One-Head
    # self.head_size = n_embed
    # 定义 self.head_size, 因为在 forward 里的scale部分需要用到
    self.head_size = head_size
    # 定义 q,k,v 的 projection
    self.q_proj = nn.Linear(n_embed, self.head_size, bias=False)
    self.k_proj = nn.Linear(n_embed, self.head_size, bias=False)
    self.v_proj = nn.Linear(n_embed, self.head_size, bias=False)
    # 当然还需要回来的projection。记错了，不需要这个。只需要concat，不需要project
    # 需要的是用register_buffer tirl (T, T)
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

  def forward(self, idx):
    # 这里负责 self-attention 的计算
    # 1. project q, k, v (B, T, n_embed) -> (B, T, head_size)
    B, T, n_embed = idx.shape
    q = self.q_proj(idx)
    k = self.k_proj(idx)
    v = self.v_proj(idx)
    # 2. 计算weight, 别忘了scale
    # B, T, head_size @ B, head_size, T => B, T, T
    weight = q @ k.transpose(-2, -1) * self.head_size ** -0.5
    # 3. 处理weight， 先用tril做mask，再softmax
    tril = self.tril[:T, :T]
    weight = weight.masked_fill(tril == 0, float('-inf'))
    # weight是 TxT 的矩阵，在第二个维度上（每一行）做softmax
    weight = F.softmax(weight, dim=-1)

    result = weight @ v
    return result

class MultiHeadAttention(nn.Module):
  # n_head * head_size = n_embed
  def __init__(self, n_head, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for i in range(n_head)])

  def forward(self, x):
    headResult = [head(x) for head in self.heads]
    # head shape is (B, T, head_size)
    # 在最后一个维度做cat，所以result的维度是 (B, T, n_embed)
    result = torch.cat(headResult, dim=-1)
    return result

class FeedForward(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embed, 4 * n_embed),
        nn.ReLU(),
        nn.Linear(4 * n_embed, n_embed),
    )

  def forward(self, x):
    return self.net(x)

class Block(nn.Module):
  def __init__(self, n_head, head_size):
    super().__init__()
    self.mh_attention = MultiHeadAttention(n_head, head_size)
    self.ffw = FeedForward()

  def forward(self, x):
    x = self.mh_attention(x)
    x = self.ffw(x)
    return x

# 3. 在 Model 中,把 MultiHeadAttention + FFW 替换成 Block
# 直接使用多个 Block
class BigramLanuageModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
    #self.sa_head = Head(n_embed)
    #self.mh_attention = MultiHeadAttention(n_head, head_size)
    #self.ffw = FeedForward()
    self.blocks = nn.Sequential(
        Block(n_head, head_size),
        Block(n_head, head_size),
        Block(n_head, head_size),
        Block(n_head, head_size),
    )
    self.position_embedding_table = nn.Embedding(block_size, n_embed)
    self.lm_head = nn.Linear(n_embed, vocab_size)

  def forward(self, idx, target=None):
    # idx, target 的shape是 (B, T)
    B, T = idx.shape
    token_emb = self.token_embedding_table(idx) # (B, T, C) 即 (B, T, n_embed)
    pos_emb = self.position_embedding_table(torch.arange(T)) # (T, n_embed)
    x = token_emb + pos_emb

    # x = self.sa_head(x)
    # x = self.mh_attention(x)
    # x = self.ffw(x)
    x = self.blocks(x)
    logits = self.lm_head(x)

    if target is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits_view = logits.view(B*T, C)
      target_view = target.view(B*T)
      # todo: 这里进行reshape是因为，F.cross_entropy的限制
      # A: 我们的数据的维度是 (B, T, C)，C是第三个维度，但是 F.cross_entropy 希望C是第二个维度
      loss = F.cross_entropy(logits_view, target_view)
    return logits, loss

  # input的shape是(1, 1), 代表batch=1， T=1
  def generate(self, input, max_tokens):
    result = input
    for i in range(max_tokens):
      # 虽然没用到loss，但是这里也得写，否则logits就变成(logits，loss)这个tuple了
      # 截断，因为position_embedding只能处理 blocl_size个数据
      context = result[:,-block_size:]
      logits, loss = self(context)
      # 只关注T维度的最后一个
      logits = logits[:,-1,:] # 变成(B, C)
      # 在最后一个维度，即 vocab_size的维度上做 softmax
      probs = F.softmax(logits, dim=-1)
      # 使用multinomial 进行采样，获取下一个token
      next_token = torch.multinomial(probs, num_samples=1) # (B, 1)
      # 加到最后去，变成 (B, T+1)
      result = torch.cat((result, next_token), dim=1)
    return result

# 初始化一个model，用于训练和evaluate
model = BigramLanuageModel(vocab_size)


n_head = 4


## Add and Norm
Block一加，网络的深度立马大幅增加。从训练的结果看，虽然网络的层级和参数都大幅增加了，但是loss并没有明显的降低。
是时候增加一些稳定网络的手段了。
Skip/Residual Connection 和 Normalization 是重要的两个手段，也是Transformer的重要组成部分。

In [ ]:
# 唯一修改的地方
class Block(nn.Module):
  def __init__(self, n_head, head_size):
    super().__init__()
    self.mh_attention = MultiHeadAttention(n_head, head_size)
    self.ffw = FeedForward()

  def forward(self, x):
    # x = self.mh_attention(x)
    # x = self.ffw(x)
    x = x + self.mh_attention(x)
    x = x + self.ffw(x)
    return x

In [ ]:
# Skip connection 比较容易
# 只需要在 Block 里修改，attention 和 ffw 后面分别增加一个 residual
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

# hyper-parameters
n_embed = 32
head_size = 8
n_head = n_embed // head_size
print(f'n_head = {n_head}')

class Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    # One-Head
    # self.head_size = n_embed
    # 定义 self.head_size, 因为在 forward 里的scale部分需要用到
    self.head_size = head_size
    # 定义 q,k,v 的 projection
    self.q_proj = nn.Linear(n_embed, self.head_size, bias=False)
    self.k_proj = nn.Linear(n_embed, self.head_size, bias=False)
    self.v_proj = nn.Linear(n_embed, self.head_size, bias=False)
    # 当然还需要回来的projection。记错了，不需要这个。只需要concat，不需要project
    # 需要的是用register_buffer tirl (T, T)
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

  def forward(self, idx):
    # 这里负责 self-attention 的计算
    # 1. project q, k, v (B, T, n_embed) -> (B, T, head_size)
    B, T, n_embed = idx.shape
    q = self.q_proj(idx)
    k = self.k_proj(idx)
    v = self.v_proj(idx)
    # 2. 计算weight, 别忘了scale
    # B, T, head_size @ B, head_size, T => B, T, T
    weight = q @ k.transpose(-2, -1) * self.head_size ** -0.5
    # 3. 处理weight， 先用tril做mask，再softmax
    tril = self.tril[:T, :T]
    weight = weight.masked_fill(tril == 0, float('-inf'))
    # weight是 TxT 的矩阵，在第二个维度上（每一行）做softmax
    weight = F.softmax(weight, dim=-1)

    result = weight @ v
    return result

class MultiHeadAttention(nn.Module):
  # n_head * head_size = n_embed
  def __init__(self, n_head, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for i in range(n_head)])

  def forward(self, x):
    headResult = [head(x) for head in self.heads]
    # head shape is (B, T, head_size)
    # 在最后一个维度做cat，所以result的维度是 (B, T, n_embed)
    result = torch.cat(headResult, dim=-1)
    return result

class FeedForward(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embed, 4 * n_embed),
        nn.ReLU(),
        nn.Linear(4 * n_embed, n_embed),
    )

  def forward(self, x):
    return self.net(x)

class Block(nn.Module):
  def __init__(self, n_head, head_size):
    super().__init__()
    self.mh_attention = MultiHeadAttention(n_head, head_size)
    self.ffw = FeedForward()

  def forward(self, x):
    # x = self.mh_attention(x)
    # x = self.ffw(x)
    x = x + self.mh_attention(x)
    x = x + self.ffw(x)
    return x

# 3. 在 Model 中,把 MultiHeadAttention + FFW 替换成 Block
# 直接使用多个 Block
class BigramLanuageModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
    #self.sa_head = Head(n_embed)
    #self.mh_attention = MultiHeadAttention(n_head, head_size)
    #self.ffw = FeedForward()
    self.blocks = nn.Sequential(
        Block(n_head, head_size),
        Block(n_head, head_size),
        Block(n_head, head_size),
        Block(n_head, head_size),
    )
    self.position_embedding_table = nn.Embedding(block_size, n_embed)
    self.lm_head = nn.Linear(n_embed, vocab_size)

  def forward(self, idx, target=None):
    # idx, target 的shape是 (B, T)
    B, T = idx.shape
    token_emb = self.token_embedding_table(idx) # (B, T, C) 即 (B, T, n_embed)
    pos_emb = self.position_embedding_table(torch.arange(T)) # (T, n_embed)
    x = token_emb + pos_emb

    # x = self.sa_head(x)
    # x = self.mh_attention(x)
    # x = self.ffw(x)
    x = self.blocks(x)
    logits = self.lm_head(x)

    if target is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits_view = logits.view(B*T, C)
      target_view = target.view(B*T)
      # todo: 这里进行reshape是因为，F.cross_entropy的限制
      # A: 我们的数据的维度是 (B, T, C)，C是第三个维度，但是 F.cross_entropy 希望C是第二个维度
      loss = F.cross_entropy(logits_view, target_view)
    return logits, loss

  # input的shape是(1, 1), 代表batch=1， T=1
  def generate(self, input, max_tokens):
    result = input
    for i in range(max_tokens):
      # 虽然没用到loss，但是这里也得写，否则logits就变成(logits，loss)这个tuple了
      # 截断，因为position_embedding只能处理 blocl_size个数据
      context = result[:,-block_size:]
      logits, loss = self(context)
      # 只关注T维度的最后一个
      logits = logits[:,-1,:] # 变成(B, C)
      # 在最后一个维度，即 vocab_size的维度上做 softmax
      probs = F.softmax(logits, dim=-1)
      # 使用multinomial 进行采样，获取下一个token
      next_token = torch.multinomial(probs, num_samples=1) # (B, 1)
      # 加到最后去，变成 (B, T+1)
      result = torch.cat((result, next_token), dim=1)
    return result

# 初始化一个model，用于训练和evaluate
model = BigramLanuageModel(vocab_size)


n_head = 4


## 单纯增加 add(还没加 norm)， 就让loss从2.5 左右，降到了 2.0左右。而且generate出来的100个字符也有明显改进。首先，已经能看出对话的形式，另外，生成的单词也越来越像样了。特别长的明显不像单词的情况比较少了。
```
generate result:
Ichy soord and band, shoust whe the itsend houm's him it arise filf aly.

RULOLAS EDWARD YICISSSWARUS:
And wheare in mary how, wout the-do the of acm your rop! wore: so;
And cidr in as we itthere coosshong viche his prie to'd him.
CLTUCHERS Luons man:
No; I his place bet thou helives:
Obitng.

MERMER:

HARD FIVNCLLINTAMESY:
Gom'd!

HENRYBENILORD:
Herul fir of As tovetrus, how nothers. I the thy of is and rofad to cown this him, if, the highit o; the still of shairs--EL:
SARGETII:
PULINCW:
Shat appillimed
a the by my a that oHhand thou a taklll But herey. shour pas gry for, it Rut Pand binde clier Londeran?

PPORMES:
ObI
```


## 增加 Norm，这里用的是 Layer Norm

In [ ]:
# 这里没有继承 nn.Module，因为这不是一个 module
# 真是一个数据处理
# Q： 是不是也可以作为一个module？
class LayerNorm:
  def __init__(self):
    self.gamma = torch.ones(n_embed)
    self.beta = torch.zeros(n_embed)

  def __call__(self, x):
    # (B, T, C) Layer Norm 是在最后一个维度，也就是feature的维度上做 Normalization
    x_mu = x.mean(-1, keepdim=True)
    x_gamma = x.std(-1, keepdim=True)
    # Here, epsilon = 1e-5, 是为了防止除数等于0
    x = (x - x_mu) / torch.sqrt(x_gamma + 1e-5)
    # gamma, beta 的维度都是 C(n_embed), 所以这里有一个到维度 (B, T, C)的广播
    result = self.gamma * x + self.beta
    return result

## 把Layer Norm的代码加进去,加到Block里面


In [ ]:
class Block(nn.Module):
  def __init__(self, n_head, head_size):
    super().__init__()
    self.mh_attention = MultiHeadAttention(n_head, head_size)
    self.ffw = FeedForward()
    self.att_norm = LayerNorm()
    self.ffw_norm = LayerNorm()

  def forward(self, x):
    # x = self.mh_attention(x)
    # x = self.ffw(x)
    # 注意这里的顺序：norm -> att/ffw -> add
    # 这个和论文上是不一样的
    x = x + self.mh_attention(self.att_norm(x))
    x = x + self.ffw(self.ffw_norm(x))
    return x

## 最后一个改动，在所有的 Block 外面，再加一个 LayerNorm

In [ ]:
self.blocks = nn.Sequential(
    Block(n_head, head_size),
    Block(n_head, head_size),
    Block(n_head, head_size),
    Block(n_head, head_size),
    LayerNorm(),
)

## 最后最后一个改动，dropout，在哪里加呢？
1. Attention
2. FeedForward
3. Head

## 最后最后最后一个改动，device
超参数改了以后，训练速度马上慢了许多（每个iteration就得10-20s，太慢了），小打小闹的时候没有感觉。
因此，必须使用 GPU 进行训练。改成colab提供的 T4 GPU以后，4s可以训练10个 iteration，几十倍的提升。
代码也需要对应修改
1. 初始化 device
2. get_batch里，把x,y moveto device
3. position_embedding, torch.arange(T, device=device)
4. model.to(device)
5. 生成自回归序列的初始变量，需要放到device
6. 使用 nn.LayerNorm 代替 LayerNorm, 所有 gamma/beta 参数就没有修改
```python
# B, T, C
# 这个配置的模型参数量大概在 6.44M
batch_size = 64
block_size = 256
n_embed = 384
```
最后，使用T4 GPU训练了5000个iteration，用时大概30分钟，loss降到1.5左右。

使用 v5e1 TPU, 代码显示 device=cpu
是不是这行代码只能检测GPU(cuda)，不能检测TPU
```python
device = 'cuda' if torch.cuda.is_available() else 'cpu'
```

# 完整代码如下

## 0. 初始化（import, 超参数 和 device）


In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

# hyper-parameters
# B, T, C
batch_size = 64
block_size = 256
n_embed = 384

# Transformer
head_size = 64
n_head = n_embed // head_size
dropout = 0.2

# device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device = {device}')

device = cpu


## 1. 准备测试数据 及 get_batch function, 确定 vocab_size

In [4]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print('size of text', len(text))
print(text[:100])

chars = sorted(list(set(text)))
vocab_size = len(chars)
print("".join(chars))
print(vocab_size)

stoi = {ch:i for i, ch in enumerate(chars)}
itos = {i:ch for i, ch in enumerate(chars)}
# print(stoi)
# print(itos)
encode = lambda str: [stoi[char] for char in str]
decode = lambda arr: ''.join([itos[id] for id in arr])
str = 'abc'
str_enc = encode(str)
str_dec = decode(str_enc)
print(str_enc, str_dec)

data = torch.tensor(encode(text), dtype=torch.long)

# split
n = int(0.9 * len(text))
train_split = data[:n]
val_split = data[n:]
print(len(train_split), len(val_split))

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# get_batch
def get_batch(split):
  data = train_split if split == 'train' else val_split
  # 生成随机数。
  # 随机数的范围是[0,len(data) - block_size),这样连续取出 block_size + 1 个数以后，最后一个数的范围是 [block_size + 1, len(data))
  # 取 block_size + 1 个数的原因是，为了获取最后一个x的下一位
  # 第二个参数表示返回的形态，不是一个数，而是一个shape=[batch_size]的tensor
  ix = torch.randint(len(data) - block_size, (batch_size,))
  # 取出block_size + 1个数，前面block_size个数作为x，往后错一位作为y
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1] for i in ix])
  # x,y 的shape都是 (batch_size, block_size)
  x = x.to(device)
  y = y.to(device)
  return x, y

# test code, print get_batch result by iterating x and y
xb, yb = get_batch('train')
print(f'xb.shape = {xb.shape}, yb.shape = {yb.shape}')
# for i in range(batch_size):
#   for j in range(block_size):
#     context = xb[i, :j+1]
#     target = yb[i, j]
#     print(f'when we input {context.tolist()}, target is {target}')


--2026-04-09 07:51:35--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.2’

input.txt.2         100%[===================>]   1.06M  --.-KB/s    in 0.004s  

2026-04-09 07:51:35 (255 MB/s) - ‘input.txt.2’ saved [1115394/1115394]

size of text 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65
[39, 40, 41] abc
1003854 111540
xb.shape = torch.Size([64, 256]), yb.shape = torch.Size([64, 256])


## 2. 模型完整代码

In [8]:
# Skip connection 比较容易
# 只需要在 Block 里修改，attention 和 ffw 后面分别增加一个 residual

class Head(nn.Module):
  def __init__(self, head_size):
    super().__init__()
    # One-Head
    # self.head_size = n_embed
    # 定义 self.head_size, 因为在 forward 里的scale部分需要用到
    self.head_size = head_size
    # 定义 q,k,v 的 projection
    self.q_proj = nn.Linear(n_embed, self.head_size, bias=False)
    self.k_proj = nn.Linear(n_embed, self.head_size, bias=False)
    self.v_proj = nn.Linear(n_embed, self.head_size, bias=False)
    # 当然还需要回来的projection。记错了，不需要这个。只需要concat，不需要project
    # 需要的是用register_buffer tirl (T, T)
    self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
    self.dropout = nn.Dropout(dropout)

  def forward(self, idx):
    # 这里负责 self-attention 的计算
    # 1. project q, k, v (B, T, n_embed) -> (B, T, head_size)
    B, T, n_embed = idx.shape
    q = self.q_proj(idx)
    k = self.k_proj(idx)
    v = self.v_proj(idx)
    # 2. 计算weight, 别忘了scale
    # B, T, head_size @ B, head_size, T => B, T, T
    weight = q @ k.transpose(-2, -1) * self.head_size ** -0.5
    # 3. 处理weight， 先用tril做mask，再softmax
    tril = self.tril[:T, :T]
    weight = weight.masked_fill(tril == 0, float('-inf'))
    # weight是 TxT 的矩阵，在第二个维度上（每一行）做softmax
    weight = F.softmax(weight, dim=-1)

    weight = self.dropout(weight)

    result = weight @ v
    return result

class MultiHeadAttention(nn.Module):
  # n_head * head_size = n_embed
  def __init__(self, n_head, head_size):
    super().__init__()
    self.heads = nn.ModuleList([Head(head_size) for i in range(n_head)])
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    headResult = [head(x) for head in self.heads]
    # head shape is (B, T, head_size)
    # 在最后一个维度做cat，所以result的维度是 (B, T, n_embed)
    result = torch.cat(headResult, dim=-1)
    result = self.dropout(result)
    return result

class FeedForward(nn.Module):
  def __init__(self):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(n_embed, 4 * n_embed),
        nn.ReLU(),
        nn.Linear(4 * n_embed, n_embed),
        nn.Dropout(dropout)
    )

  def forward(self, x):
    return self.net(x)

# 这里没有继承 nn.Module，因为这不是一个 module
# 只是一个数据处理
# Q： 是不是也可以作为一个module？
# A: 因为要在blocks最后加 LayerNorm，所以必须改成 nn.Module
# __call__ 也得改成 forward
# __init__ 里必须调用 super().__init__(), 否则初始化AdamW的时候，取不到 module.parameters()
class LayerNorm(nn.Module):
  def __init__(self):
    super().__init__()
    self.gamma = torch.ones(n_embed).to(device)
    self.beta = torch.zeros(n_embed).to(device)

  def forward(self, x):
    # (B, T, C) Layer Norm 是在最后一个维度，也就是feature的维度上做 Normalization
    x_mu = x.mean(-1, keepdim=True)
    x_gamma = x.std(-1, keepdim=True)
    # Here, epsilon = 1e-5, 是为了防止除数等于0
    x = (x - x_mu) / torch.sqrt(x_gamma + 1e-5)
    # gamma, beta 的维度都是 C(n_embed), 所以这里有一个到维度 (B, T, C)的广播
    result = self.gamma * x + self.beta
    return result

class Block(nn.Module):
  def __init__(self, n_head, head_size):
    super().__init__()
    self.mh_attention = MultiHeadAttention(n_head, head_size)
    self.ffw = FeedForward()
    self.att_norm = LayerNorm()
    self.ffw_norm = LayerNorm()

  def forward(self, x):
    # x = self.mh_attention(x)
    # x = self.ffw(x)
    x = x + self.mh_attention(self.att_norm(x))
    x = x + self.ffw(self.ffw_norm(x))
    return x

# 3. 在 Model 中,把 MultiHeadAttention + FFW 替换成 Block
# 直接使用多个 Block
class BigramLanuageModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__()
    self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
    #self.sa_head = Head(n_embed)
    #self.mh_attention = MultiHeadAttention(n_head, head_size)
    #self.ffw = FeedForward()
    self.blocks = nn.Sequential(
        Block(n_head, head_size),
        Block(n_head, head_size),
        Block(n_head, head_size),
        Block(n_head, head_size),
        LayerNorm(),
    )
    self.position_embedding_table = nn.Embedding(block_size, n_embed)
    self.lm_head = nn.Linear(n_embed, vocab_size)

  def forward(self, idx, target=None):
    # idx, target 的shape是 (B, T)
    B, T = idx.shape
    token_emb = self.token_embedding_table(idx) # (B, T, C) 即 (B, T, n_embed)
    pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, n_embed)
    x = token_emb + pos_emb

    # x = self.sa_head(x)
    # x = self.mh_attention(x)
    # x = self.ffw(x)
    x = self.blocks(x)
    logits = self.lm_head(x)

    if target is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits_view = logits.view(B*T, C)
      target_view = target.view(B*T)
      # todo: 这里进行reshape是因为，F.cross_entropy的限制
      # A: 我们的数据的维度是 (B, T, C)，C是第三个维度，但是 F.cross_entropy 希望C是第二个维度
      loss = F.cross_entropy(logits_view, target_view)
    return logits, loss

  # input的shape是(1, 1), 代表batch=1， T=1
  def generate(self, input, max_tokens):
    result = input
    for i in range(max_tokens):
      # 虽然没用到loss，但是这里也得写，否则logits就变成(logits，loss)这个tuple了
      # 截断，因为position_embedding只能处理 blocl_size个数据
      context = result[:,-block_size:]
      logits, loss = self(context)
      # 只关注T维度的最后一个
      logits = logits[:,-1,:] # 变成(B, C)
      # 在最后一个维度，即 vocab_size的维度上做 softmax
      probs = F.softmax(logits, dim=-1)
      # 使用multinomial 进行采样，获取下一个token
      next_token = torch.multinomial(probs, num_samples=1) # (B, 1)
      # 加到最后去，变成 (B, T+1)
      result = torch.cat((result, next_token), dim=1)
    return result

# 初始化一个model，用于训练和evaluate
model = BigramLanuageModel(vocab_size)
model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')


6.644033 M parameters


## 3. 训练

In [9]:
learning_rate = 1e-4
iterations = 5000

optimizer = torch.optim.AdamW(model.parameters(), learning_rate)

for i in range(iterations):
  xb, yb = get_batch('train')
  optimizer.zero_grad()
  logits, loss = model(xb, yb)
  loss.backward()
  optimizer.step()
  if i % 10 == 0:
    print(f'Step = {i}, loss={loss}')

Step = 0, loss=4.417713165283203
Step = 10, loss=3.138643264770508
Step = 20, loss=2.9617881774902344
Step = 30, loss=2.8697381019592285
Step = 40, loss=2.726174831390381
Step = 50, loss=2.6860060691833496
Step = 60, loss=2.6482350826263428
Step = 70, loss=2.61611008644104
Step = 80, loss=2.6191294193267822
Step = 90, loss=2.582989454269409
Step = 100, loss=2.554946184158325
Step = 110, loss=2.5372185707092285
Step = 120, loss=2.5527100563049316
Step = 130, loss=2.5254616737365723
Step = 140, loss=2.5178213119506836
Step = 150, loss=2.5270307064056396
Step = 160, loss=2.52710223197937
Step = 170, loss=2.5233042240142822
Step = 180, loss=2.504519462585449
Step = 190, loss=2.4841091632843018
Step = 200, loss=2.5008959770202637
Step = 210, loss=2.4879393577575684
Step = 220, loss=2.480173349380493
Step = 230, loss=2.459591865539551
Step = 240, loss=2.493131637573242
Step = 250, loss=2.5160536766052246
Step = 260, loss=2.4813573360443115
Step = 270, loss=2.493051528930664
Step = 280, loss=

## 4. 检查成果，自回归生成连续文本

In [10]:
# 调用generate函数，生成连续文本
input = torch.zeros((1, 1), dtype=torch.long, device=device)
max_tokens = 1000
out = model.generate(input, max_tokens)
# Q：为什么要取 out[0]? generate返回值的格式是什么？
# A: input的格式是(B, T), generate的返回值格式也是 (B, T)，虽然B=1, 但是也占着一个维度
# 所以out[0]的意思就是，把唯一的这个sample取出来
out_str = decode(out[0].tolist())
print(f'generate result: {out_str}')


generate result: 
Wilt stare: I mustied respring to any chase
I mokervelimable by my witton of me again; bless,
Pets be a mide dear for in mothersheit must.

DUKE VINCENTIO:
For his grom mostand me a stard? ore ruice, 'twillis.

DUKE VINCENENTIO:
Would not hall never the and hange will beliar.

Provost:
Besise shem time grave for the palking.

QUEEN MARGARET:
Gretake away that I leave and as your braise
Where will his wearthd, nitter store.

KING RIV:
Hold:
If you'll upon the behen, you keir men your
Till'd, forberty. Wherit where you shall own.

CANTAMER:
So his peaking anotio. Hereby the world!
Who empart, as grief Marcy, then thy chile.
If not dost I am thou stand at thy hearts;
But they fance charms called them at curse
Upon Grouch patier, what is
So he have cannect with nain's it a crazen,
Beek till prace. Farius beasted on our compand.
3ward me no they, nare.

POLIXENES:
Though I gime love me speak, and this did he littong
gallordlames to more I couse,--my hast
till enour: out di